# Leffa try-on server on Kaggle

**Settings → Accelerator: GPU T4 x2 → Internet: On** before running.

Run CELL 1, wait for `Running on public URL`, then run CELL 2 with that URL.
Only wire the URL into the app if CELL 2 prints **REAL RENDER**.

Keep this tab open — closing it kills the tunnel.


In [ ]:
# ============================================================================
# CELL 1 — Leffa on Kaggle.  Settings: Accelerator = GPU T4 x2, Internet = On
# ============================================================================
import subprocess, sys, os, types, re, gc, threading, time

print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU — set Accelerator first")
print("host RAM GB:", round(os.sysconf("SC_PAGE_SIZE")*os.sysconf("SC_PHYS_PAGES")/1e9, 1))

# onnxruntime is NOT optional: human parsing runs parsing_lip.onnx / parsing_atr.onnx.
# It is in Leffa's own requirements.txt and omitting it fails at IMPORT, after the
# whole download has completed.
#
# huggingface_hub is deliberately NOT pinned. A 0.15.1 pin predates the
# snapshot_download behaviour this notebook depends on.
!pip -q install --upgrade "gradio>=4" gradio-client diffusers transformers accelerate \
    peft safetensors einops omegaconf opencv-python scikit-image timm huggingface_hub \
    av fvcore cloudpickle pycocotools torchmetrics onnxruntime 2>&1 | tail -3

# @spaces.GPU is Hugging Face's own runtime shim and does not exist off their
# infrastructure. A no-op decorator is the entire requirement.
stub = types.ModuleType("spaces")
def GPU(*a, **k):
    def deco(fn): return fn
    return deco(a[0]) if a and callable(a[0]) else deco
stub.GPU = GPU
sys.modules["spaces"] = stub

# KAGGLE PATHS. /content is Colab; on Kaggle it is /kaggle/working, which is
# also the directory the 19.5 GiB output cap applies to.
WORK = "/kaggle/working/leffa"
import shutil; shutil.rmtree(WORK, ignore_errors=True)
!git lfs install -q 2>/dev/null
!git clone -q https://huggingface.co/spaces/franciszzj/Leffa {WORK}
os.chdir(WORK)
src = open("app.py").read()

# ---------------------------------------------------------------- the patch
# app.py builds THREE diffusion models at import:
#   vt_model_hd  SD1.5 inpainting   try-on, VITON-HD      7.21 GB   KEEP
#   vt_model_dc  SD1.5 inpainting   try-on, DressCode     7.21 GB   KEEP
#   pt_model     STABLE DIFFUSION XL  pose transfer      20.88 GB   DROP
#
# pt_model is larger than the other two combined and drives POSE TRANSFER — a
# feature unrelated to try-on that this project never calls. Dropping only it
# keeps BOTH try-on models, so viton_hd AND dress_code work and lower-body and
# dresses still render.
#
# 36.84 GB -> 15.97 GB, which fits Kaggle's 19.5 GiB cap with ~5 GB spare.

# 1. Do not DOWNLOAD it. snapshot_download runs before the model lines, so
#    without this the 20.88 GB arrives regardless of what is loaded.
src = src.replace(
    'snapshot_download(repo_id="franciszzj/Leffa", local_dir="./ckpts")',
    'snapshot_download(repo_id="franciszzj/Leffa", local_dir="./ckpts",'
    ' ignore_patterns=["*pose_transfer*", "*stable-diffusion-xl*"])')

# 2. Do not LOAD it. `pt_inference` is ALIASED, not set to None: app.py's UI
#    references it at import time. Lazy `.*?`, not greedy `.*`.
src = re.sub(r"pt_model = LeffaModel\(.*?\)\s*pt_inference = LeffaInference\(model=pt_model\)",
             "pt_inference = vt_inference_hd  # pose transfer dropped", src, flags=re.S)

# Fail LOUDLY if either patch missed. A silent no-op here means 36.84 GB into a
# 19.5 GB cap, discovered twenty minutes later.
assert "ignore_patterns" in src, "download filter did not apply"
assert src.count("LeffaModel(") == 2, "load patch did not apply — app.py changed upstream"
print("\npatched: pose-transfer skipped (download AND load) -> 15.97 GB of 36.84 GB")
print("both try-on models kept: viton_hd + dress_code\n")

# ------------------------------------------------------- progress reporting
# In a background thread because Jupyter QUEUES cells: a second cell cannot run
# while this one is executing, so a separate monitor cell would never start.
TARGET_GB, CAP_GB = 15.97, 20.94   # measured from the HF API; 19.5 GiB cap

def _watch():
    last, last_t = 0.0, time.time()
    while not getattr(_watch, "stop", False):
        gb = 0.0
        if os.path.isdir("./ckpts"):
            gb = sum(os.path.getsize(os.path.join(r, f))
                     for r, _, fs in os.walk("./ckpts")
                     for f in fs if os.path.exists(os.path.join(r, f))) / 1e9
        now = time.time()
        rate = (gb - last) / max(now - last_t, 1e-6)
        pct  = min(100.0, gb / TARGET_GB * 100)
        bar  = "#" * int(pct // 4) + "." * (25 - int(pct // 4))
        left = CAP_GB - gb
        print(f"  [{time.strftime('%H:%M:%S')}] [{bar}] {pct:5.1f}%  {gb:5.2f}/{TARGET_GB} GB  "
              f"{rate*1000:6.1f} MB/s  ETA {((TARGET_GB-gb)/max(rate,1e-9))/60:4.1f}m  "
              f"disk left {left:5.2f} GB" + ("   <-- NEAR CAP" if left < 1.5 else ""), flush=True)
        last, last_t = gb, now
        time.sleep(20)

threading.Thread(target=_watch, daemon=True).start()
gc.collect()
try:
    exec(src)            # prints: Running on public URL: https://xxxx.gradio.live
finally:
    _watch.stop = True


In [ ]:
# ============================================================================
# CELL 2 — prove it is NOT a passthrough.  Run while CELL 1 is still serving.
# ============================================================================
# A widely-shared "memory-optimized" notebook for this model contains
# `def lightweight_tryon(person, garment): return person` — it builds the right
# endpoint shape and hands back the person photo unchanged. Everything looks
# like it works. Only the pixels tell you otherwise.
import io, requests
from PIL import Image, ImageChops
import numpy as np
from gradio_client import Client, handle_file

URL = "https://PASTE-YOUR-URL.gradio.live"     # <-- from CELL 1's output

person  = "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/example/human/00034_00.jpg"
garment = "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/example/cloth/04469_00.jpg"

c = Client(URL)
eps = list(c.view_api(return_format="dict")["named_endpoints"])
print("endpoints:", eps)
assert "/leffa_predict_vt" in eps, "wrong app running — expected the real Leffa endpoint"

out = c.predict(
    src_image_path=handle_file(person),
    ref_image_path=handle_file(garment),
    ref_acceleration="False", step=30, scale=2.5, seed=42,
    vt_model_type="viton_hd", vt_garment_type="upper_body", vt_repaint="False",
    api_name="/leffa_predict_vt",
)
result = Image.open(out[0] if isinstance(out, (list, tuple)) else out).convert("RGB")
source = Image.open(io.BytesIO(requests.get(person, timeout=60).content)).convert("RGB")

diff = np.asarray(ImageChops.difference(source.resize(result.size), result), float).mean()
print(f"\nmean pixel difference vs input: {diff:.1f} / 255")
print("PASSTHROUGH — the model is NOT running" if diff < 2 else "REAL RENDER — safe to wire up")
display(result)
